# Pneumonia detection in chest X-rays: notebook walkthrough

Companion to `train.py`. Same pipeline, step by step, with visualizations. Run from the repository root after `pip install -r requirements.txt`.

## 1. Load the data

PneumoniaMNIST (MedMNIST v2, Zenodo): 28x28 grayscale chest X-rays. `train.download_data()` caches the npz under `data/` on first run.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from train import download_data, load_split
from model import PneumoniaCNN

path = download_data()
train_ds, val_ds, test_ds = (load_split(path, s) for s in ('train', 'val', 'test'))
print(len(train_ds), len(val_ds), len(test_ds))

## 2. Quick EDA

Class balance and a look at what the model sees. The training split is roughly 3:1 pneumonia to normal, which motivates the `pos_weight` in the loss.

In [ ]:
labels = train_ds.tensors[1].numpy()
names = ['normal', 'pneumonia']
counts = [(labels == k).sum() for k in (0, 1)]
print(dict(zip(names, counts)), 'neg/pos =', counts[0] / counts[1])

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
rng = np.random.default_rng(0)
for row, cls in enumerate((0, 1)):
    idx = rng.choice(np.where(labels == cls)[0], 8, replace=False)
    for ax, i in zip(axes[row], idx):
        ax.imshow(train_ds.tensors[0][i, 0], cmap='gray')
        ax.axis('off')
        ax.set_title(names[cls], fontsize=8)
plt.tight_layout()

## 3. Model

Three conv blocks with batch norm, global average pooling, one linear layer. ~93k parameters, deliberately small for 4.7k training images.

In [ ]:
model = PneumoniaCNN()
n_params = sum(p.numel() for p in model.parameters())
print(f'{n_params:,} parameters')
model

## 4. Train

`python train.py --epochs 6` runs the full loop and writes artifacts to `results/`. On a CPU this takes about two minutes, so it is usually easier to run the script than to duplicate the loop here.

In [ ]:
# Equivalent to: python train.py --epochs 6 --outdir results
# (kept as a comment so the notebook stays fast to execute end-to-end)

## 5. Results

Load the metrics and figures produced by the training run.

In [ ]:
import json
from PIL import Image

with open('results/metrics.json') as f:
    metrics = json.load(f)
print(json.dumps(metrics, indent=2))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, name in zip(axes, ['training_curves', 'confusion_matrix', 'sample_predictions']):
    ax.imshow(Image.open(f'results/{name}.png'))
    ax.axis('off')
    ax.set_title(name)
plt.tight_layout()

## 6. Takeaways

- Test accuracy ~0.89, ROC-AUC ~0.96 on a real (if downsampled) medical task.
- Weighted BCE handles the 3:1 imbalance without resampling.
- Next: transfer learning on full-resolution images, threshold tuning for target recall, calibration.